In [1]:
import pandas as pd

df = pd.read_excel('../data/raw/main.xlsx')

#پاکسازی دیتای نمره و تبدیل نمره
def Score_conversion(x):
    try:
        return float(x)
    except:
        return None

df['نمره_عددی'] = df['نمره'].apply(Score_conversion)
print("ready")

ready


In [2]:
profile = df.groupby('شماره‌دانشجويي').agg(
    میانگین_نمره = ('نمره_عددی' , 'mean'),
    تعداد_درس = ('نمره_عددی' , "count") , 
    تعداد_مردودی = ('نمره_عددی' , lambda x: (x < 10).sum()),
    تعداد_فاقد_برگه = ('نمره', lambda x:(x == 'فاقد برگه').sum()),

).reset_index()

In [4]:
print(profile.shape)
profile.head(10)

(7769, 5)


,شماره‌دانشجويي,میانگین_نمره,تعداد_درس,تعداد_مردودی,تعداد_فاقد_برگه
0,110095300001,NaN,0,0,6
1,110095300003,NaN,0,0,7
2,110095300004,11.882857,35,9,13
3,110095300005,14.644737,38,1,0
4,110095300007,11.455000,50,14,1
5,110095300009,13.250000,9,1,4
6,110095300010,9.221154,26,11,2
7,110095300011,15.444444,9,1,15
8,110095300012,13.610000,25,4,3
9,110095300014,NaN,0,0,7


In [8]:
# اگه وضعیت توی لیست خطر بود True بذار
risk_status = [
    'حذف ترم باسنوات',
    'حذف ترم بدون سنوات',
    'محروم از تحصيل با احتساب در سنوات',
    'تعليق ترم با احتساب در سنوات',
    'تعليق ترم بدون احتساب در سنوات'
]

df['خطرناک'] = df['وضعيت‌ترم‌دانشجو'].isin(risk_status)


In [6]:
print(df.columns.tolist())

['کدترم امتحان', 'شماره\u200cدانشجويي', 'کد\u200cترم\u200cورود دانشجو', 'درس', 'کددرس', 'نمره', 'نوع\u200cنمره', 'کد\u200cگروه\u200cدرسي', 'کدملي\u200cمدرس', 'نام\u200cمدرس', 'وضعيت\u200cترم\u200cدانشجو', 'شناسه\u200cدانشكده', 'نمره_عددی']


In [13]:
student_risk = df.groupby('شماره‌دانشجويي')['خطرناک'].max()

In [15]:
profile['در_معرض_خطر'] = profile['شماره‌دانشجويي'].map(student_risk).astype(int)

profile['در_معرض_خطر'].value_counts()

در_معرض_خطر
0    7564
1     205
Name: count, dtype: int64

In [16]:
profile['در_معرض_خطر'] = (
    (profile['در_معرض_خطر'] == 1) |
    (profile['میانگین_نمره'] < 12) |
    (profile['تعداد_مردودی'] > 3)
).astype(int)

profile['در_معرض_خطر'].value_counts()

در_معرض_خطر
0    5569
1    2200
Name: count, dtype: int64

In [18]:
profile.to_excel('../data/processed/پروفایل_دانشجو.xlsx', index=False)